# AI Programming Foundations: Titanic Data Workflow

**Author:** Francisco Mateus

**Dataset:** Titanic: Machine Learning from Disaster (Kaggle)

This notebook builds a reproducible workflow over the 891-row Titanic passenger
table. It measures what is missing, cleans the table with four pure functions,
explores survival by sex, class, title and deck, draws five labelled figures,
and closes with a summary.

## How this notebook is organised

The headings follow the project's workflow, with Assessment and Cleaning as
separate sections:

1. **Setup**: the libraries and the versions this run used.
2. **Ingestion**: the CSV loaded from disk, with its provenance.
3. **Assessment**: what is missing, and where the gaps sit.
4. **Cleaning**: the functions that repair it, each one pure.
5. **Exploratory Analysis**: survival rates by sex, class, title and deck.
6. **Visualizations**: the labelled figures, each interpreted in the cell below it.
7. **Summary**: what the numbers support and what they do not.

Every number quoted in the written report is printed by a cell in this notebook.


## 1. Setup

I placed all the imports and the display settings in this one cell for clear organisation, and I also printed the four package versions.


In [1]:
# One import cell so no later cell can change the display settings quietly
# The version line is the reproducibility record for this run
%matplotlib inline

import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")


numpy 2.5.3 | pandas 3.0.6 | matplotlib 3.11.2 | seaborn 0.13.2


## 2. Ingestion

Following the project instructions, I placed the dataset in the same folder as the notebook, 
so the notebook reads the local `titanic.csv`.
The source URL is recorded in a comment in the code cell below.


In [2]:
# Provenance: Kaggle "Titanic - Machine Learning from Disaster",
# mirror https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
# The CSV ships beside this notebook (project Task 1)
df = pd.read_csv("titanic.csv")

df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### What the rows mean

The table below is the column dictionary: later sections check values against it.

| Column | Kind | Meaning |
|---|---|---|
| `PassengerId` | number | Row identifier; it says nothing about the person |
| `Survived` | `0` or `1` | The outcome to explain: `1` survived, `0` did not |
| `Pclass` | `1`, `2`, `3` | Ticket class: `1` first, `2` second, `3` third, there is no fourth class |
| `Name` | text | Full name; carries the honorific that the Cleaning section turns into a `title` feature |
| `Sex` | text | `male` or `female` |
| `Age` | number | Age in years; blanks are counted in the Assessment section |
| `SibSp` | number | Siblings and spouses aboard |
| `Parch` | number | Parents and children aboard |
| `Ticket` | text | Ticket number |
| `Fare` | number | Price paid for the ticket |
| `Cabin` | text | Cabin code; blank for many rows |
| `Embarked` | text | Port of boarding: `S` Southampton, `C` Cherbourg, `Q` Queenstown |


### Load check

I will check if the table has the following:

- `891` rows and `12` columns, the size the project expects;
- numeric columns stored as numbers, `int64` or `float64`;
- text columns stored as text, `str`.

The next cell prints each check and then asserts the shape and the dtypes, so a different file fails the run instead of producing plausible numbers.


In [3]:
# Note: Dtypes and shape asserted so a different file or an older pandas stops the run here
print(f"rows x columns: {df.shape}")
print(f"columns: {list(df.columns)}")
print(df.dtypes.to_frame("dtype").to_string())

number_columns = ["PassengerId", "Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
text_columns = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]

for col in number_columns:
    assert df[col].dtype.name in {"int64", "float64"}, f"{col} stored as {df[col].dtype.name}"
for col in text_columns:
    assert df[col].dtype.name == "str", f"{col} stored as {df[col].dtype.name}"
print(f"dtype check: {len(number_columns)} numeric, {len(text_columns)} str")

assert df.shape == (891, 12), f"unexpected shape: {df.shape}"
print("load check: 891 rows, 12 columns")


rows x columns: (891, 12)
columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
               dtype
PassengerId    int64
Survived       int64
Pclass         int64
Name             str
Sex              str
Age          float64
SibSp          int64
Parch          int64
Ticket           str
Fare         float64
Cabin            str
Embarked         str
dtype check: 7 numeric, 5 str
load check: 891 rows, 12 columns


## 3. Assessment

In this section I measure the data before cleaning, and I answer four questions:

1. How many values are missing in each column?
2. Where do the gaps sit?
3. Does any row repeat another?
4. Are the values valid (plausible `Age` and `Fare` ranges, each coded
   column holding only the values the column dictionary lists)?

Each check ends in an assertion, so the run stops on a different file instead of passing silently.


### Missing values

For this check, I counted the blanks in every column, including the columns with no missing values, and printed each count beside its percentage of the `891` rows, because `2` missing values and `177` missing values are problems of different sizes.

The cell also prints the two rows whose `Embarked` is blank, the only gap small enough to check row by row. The assertions pin the three gaps this file carries.


In [4]:
# Missing-value survey of the raw table, each count beside its share of the 891 rows
missing = df.isna().sum()
pct_missing = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame(
    {"missing": missing, "pct_missing": pct_missing}
).sort_values("missing", ascending=False, kind="stable")
print(missing_report.to_string())

gapped = missing[missing > 0]
print(f"\ncolumns with gaps: {len(gapped)} -> {list(gapped.index)}")
print(f"complete columns: {len(missing) - len(gapped)}")

# The only gap small enough to inspect: one ticket, one cabin, the port missing twice
print("\nrows missing Embarked:")
print(df.loc[df["Embarked"].isna(), ["Name", "Ticket", "Fare", "Cabin"]]
      .to_string(index=False))

# Assert the three gaps this file is known to carry
assert missing["Age"] == 177, f"Age gaps: {missing['Age']}"
assert missing["Cabin"] == 687, f"Cabin gaps: {missing['Cabin']}"
assert missing["Embarked"] == 2, f"Embarked gaps: {missing['Embarked']}"
print("\ngap check: Age 177, Cabin 687, Embarked 2")


             missing  pct_missing
Cabin            687        77.10
Age              177        19.87
Embarked           2         0.22
PassengerId        0         0.00
Survived           0         0.00
Pclass             0         0.00
Name               0         0.00
Sex                0         0.00
SibSp              0         0.00
Parch              0         0.00
Ticket             0         0.00
Fare               0         0.00

columns with gaps: 3 -> ['Age', 'Cabin', 'Embarked']
complete columns: 9

rows missing Embarked:
                                     Name Ticket  Fare Cabin
                      Icard, Miss. Amelie 113572  80.0   B28
Stone, Mrs. George Nelson (Martha Evelyn) 113572  80.0   B28

gap check: Age 177, Cabin 687, Embarked 2


### Duplicates and validity

A table can be complete and still broken, so I checked three properties beyond the blanks:

- **Uniqueness**: no row is a duplicate, and every `PassengerId` is distinct.
- **Categories**: I counted every coded column and asserted it holds only the values the column dictionary lists.
- **Ranges**: I summarised `Age` and `Fare`, so the min and max are visible in the printed summary.

A fare of `0` is unusual but not impossible, so I counted those rows and kept them as they are.


In [5]:
# A table can be complete and still broken, so checks beyond the blanks
dup_rows = df.duplicated().sum()
dup_ids = df["PassengerId"].duplicated().sum()
print(f"duplicate rows: {dup_rows}")
print(f"duplicate PassengerId: {dup_ids}")

assert dup_rows == 0, "duplicate rows found"
assert df["PassengerId"].is_unique, "PassengerId repeats"
print("uniqueness check: 891 rows, 891 distinct PassengerId values")

# value_counts drops NaN, so the Embarked counts add to 889; the two blanks
# were already counted in the missing-values cell
for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"\n{col}:")
    print(df[col].value_counts())

# Hold every coded column to the column dictionary
assert set(df["Survived"].unique()) == {0, 1}
assert set(df["Pclass"].unique()) == {1, 2, 3}
assert set(df["Sex"].unique()) == {"male", "female"}
assert set(df["Embarked"].dropna().unique()) == {"S", "C", "Q"}
print("\ncategory check: only codebook values present")

print()
# describe() counts non-empty values, so the Age count cross-checks the 177 blanks
print(df[["Age", "Fare"]].describe().round(4).to_string())

# Zero fares: counted for the record, then left as they are
zero_fare = df.loc[df["Fare"] == 0]
assert len(zero_fare) == 15, f"zero fares: {len(zero_fare)}"
print(f"\nzero fares: {len(zero_fare)}")
print(zero_fare["Pclass"].value_counts())
print("validity check: 15 zero fares, left as they are")


duplicate rows: 0
duplicate PassengerId: 0
uniqueness check: 891 rows, 891 distinct PassengerId values

Survived:
Survived
0    549
1    342
Name: count, dtype: int64

Pclass:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Sex:
Sex
male      577
female    314
Name: count, dtype: int64

Embarked:
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

category check: only codebook values present

            Age      Fare
count  714.0000  891.0000
mean    29.6991   32.2042
std     14.5265   49.6934
min      0.4200    0.0000
25%     20.1250    7.9104
50%     28.0000   14.4542
75%     38.0000   31.0000
max     80.0000  512.3292

zero fares: 15
Pclass
2    6
1    5
3    4
Name: count, dtype: int64
validity check: 15 zero fares, left as they are


### What the assessment found

The findings below set up the cleaning section:

- `Age` is blank in `177` rows (`19.87%` of the table). That is too many to drop, so the blanks need filling.
- `Cabin` is blank in `687` rows (`77.10%`). The column is mostly absent, so the part that still carries information is the deck letter.
- `Embarked` is blank in `2` rows. Both rows share ticket `113572` and cabin `B28`, so one booking produced both gaps and one fill can fix both.
- `Fare` reaches `0` in `15` rows. They look like real records, so I left them as they are.

The other checks found nothing wrong: no duplicate rows, no repeated `PassengerId`, plausible `Age` and `Fare` ranges, and only the values the column dictionary lists.
